# Crop to AOI

## Import libs

In [ ]:
import os
import glob
import geopandas as gpd
import rioxarray as rxr
import matplotlib.pyplot as plt
from pathlib import Path
import rasterio
import numpy as np

## Define File Paths

In [ ]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

In [ ]:
input_folder = os.path.join(base_path, "5_Added_CHM_Channel")
cropped_folder = os.path.join(base_path, "6_Cropped_Images")
os.makedirs(cropped_folder, exist_ok=True)
roi_path = os.path.join(input_folder, "final_roi.shp")

## Load ROI Data

In [ ]:
roi = gpd.read_file(roi_path)

print(f"ROI loaded. CRS: {roi.crs}")

roi.plot(edgecolor="red", facecolor="none", linewidth=2, figsize=(8, 6))
plt.title("Region of Interest (ROI) - Cutout Area")
plt.show()

## Cut all TIFs to ROI

In [ ]:
NODATA_VAL = -9999.0

# 1. Get all TIFs
all_tifs = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found TIF-Files to cut: {len(all_tifs)}\n")

cropped_files_paths = []

for file in all_tifs:

    if "250821" not in file:
        print("Pass")
        continue
    
    basename = os.path.basename(file)
    print(f"Processing: {basename}...")
    
    # A) Load Raster image
    ds = rxr.open_rasterio(file)
    
    # B) CRS Check
    raster_crs = ds.rio.crs
    
    if roi.crs != raster_crs:
        print(f"  -> ⚠️ CRS Mismatch! Raster: {raster_crs} | ROI: {roi.crs}")
        print("  -> Transform shapefile to Raster-CRS...")
        roi_current = roi.to_crs(raster_crs)
    else:
        roi_current = roi

    # Plot before
    plt.clf()
    ds.isel(band=slice(0, 3)).clip(0, 255).astype("uint8").plot.imshow()
    plt.title(f"Raster before cropping - {basename}")
    plt.show()
        
    # C) Cropping
    try:
        ds.rio.write_nodata(NODATA_VAL, inplace=True)
        ds_cropped = ds.rio.clip(roi_current.geometry, roi_current.crs, drop=True)

        data = ds_cropped.values
        for i in range(3):  # Nur Kanal 0, 1, 2 (RGB)
            valid_mask = data[i] != NODATA_VAL
            data[i][valid_mask] = np.clip(data[i][valid_mask], 0, 255)
        ds_cropped.values = data

        # Plot after
        plt.clf()
        ds_cropped.isel(band=slice(0, 3)).clip(0, 255).astype("uint8").plot.imshow()
        plt.title(f"Raster AFTER cropping - {basename}")
        plt.show()
        
        # D) Save
        output_path = os.path.join(cropped_folder, basename.replace(".tif", "_cropped.tif"))
        ds_cropped.rio.write_nodata(NODATA_VAL, inplace=True)
        ds_cropped.rio.to_raster(output_path)
        cropped_files_paths.append(output_path)
        print("  -> ✅ Successfully cropped and saved!\n")
        
    except Exception as e:
        print(f"  -> ❌ Error during cropping of {basename}: {e}\n")
        
    finally:
        ds.close()
        if 'ds_cropped' in locals():
            ds_cropped.close()

print("All images were processed!")

In [ ]:
print("==================================================")
print("✂️ SANITY CHECK: CROPPED BORDERS")
print("==================================================\n")

# Check cropped files
check_files = glob.glob(os.path.join(cropped_folder, "*_cropped.tif"))

for file in check_files:
    basename = os.path.basename(file)
    print(f"--- Checking: {basename} ---")
    
    with rasterio.open(file) as src:
        # 1. Metadata Check
        meta_nodata = src.nodata
        
        # 2. Physical Pixel Check (Only read first channel)
        data = src.read(1)
        
        # Count number of nodata values
        nodata_pixel_count = np.sum(data == NODATA_VAL)
        
        # Count if there are NaNs
        nan_count = np.sum(np.isnan(data))
        
        # Filter valid pixels (not -9999.0)
        valid_pixels = data[data != NODATA_VAL]
        
        # Check true min value
        true_min = np.min(valid_pixels) if valid_pixels.size > 0 else "N/A"
        
        # --- EVALUATION ---
        print(f"  Metadata NoData Tag: {'✅ ' + str(meta_nodata) if meta_nodata == NODATA_VAL else '❌ ' + str(meta_nodata)}")
        print(f"  Found physical -9999.0 pixels? {'✅ Yes (' + str(nodata_pixel_count) + ' Pixels)' if nodata_pixel_count > 0 else '❌ No (0 Pixels)'}")
        print(f"  Hidden NaNs present? {'✅ No' if nan_count == 0 else '❌ Ja (' + str(nan_count) + ' Pixels)'}")
        
        if true_min != "N/A" and true_min >= 0:
            print(f"  True Minimum Pixel Value: ✅ {true_min:.2f} (No 0.0 Error)")
        else:
            print(f"  True Minimum Pixel Valuet: ⚠️ {true_min} (Take care, negative Value found!)")
        
    print("\n")